# Model Improvement and Prediction

This notebook covers the final two practical stages of the project:

1. **Model improvement**: fine-tune the best model from model selection.
2. **Prediction**: save the final model and create a reusable inference function.

The model used here is **LightGBM Regressor** because it produced the best validation WAPE during model selection.

## 1. Import libraries and define paths

In [1]:
from pathlib import Path
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


def find_project_dir(start_path=None):
    """Find the project root by walking upward until the prepared parquet file is found."""
    current = Path.cwd() if start_path is None else Path(start_path).resolve()
    for candidate in [current, *current.parents]:
        prepared = candidate / "data" / "processed" / "hourly_taxi_demand_2023_prepared.parquet"
        if prepared.exists():
            return candidate
    raise FileNotFoundError("Could not find data/processed/hourly_taxi_demand_2023_prepared.parquet")


PROJECT_DIR = find_project_dir()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PREPARED_PATH = PROCESSED_DIR / "hourly_taxi_demand_2023_prepared.parquet"
TUNING_RESULTS_PATH = PROCESSED_DIR / "lgbm_tuning_results.csv"
MODEL_ARTIFACT_PATH = PROCESSED_DIR / "final_lgbm_trip_demand_model.joblib"
TEST_PREDICTIONS_PATH = PROCESSED_DIR / "december_lgbm_predictions.parquet"

print(f"Project directory: {PROJECT_DIR}")
print(f"Prepared data: {PREPARED_PATH}")

Project directory: c:\Users\USER\Downloads\calibrax\forecasting-taxi-demand-across-nyc-zones
Prepared data: c:\Users\USER\Downloads\calibrax\forecasting-taxi-demand-across-nyc-zones\data\processed\hourly_taxi_demand_2023_prepared.parquet


## 2. Load prepared data

The input is the output from data_prep.ipynb. It already contains cleaned hourly zone demand, weather, holiday, calendar, lag, and rolling features.

In [2]:
df = pd.read_parquet(PREPARED_PATH)

print(df.shape)
print(df["pickup_hour"].min(), "to", df["pickup_hour"].max())
df.head(10)

(2303880, 51)
2023-01-01 00:00:00 to 2023-12-31 23:00:00


,pickup_hour,PULocationID,trip_count,revenue,avg_trip_distance,avg_fare_amount,avg_trip_duration_min,zero_passenger_count,Borough,Zone,service_zone,weather_station_key,weather_station_name,STATION,usaf,wban,temp_c,dew_c,pressure_hpa,wind_speed_mps,visibility_m,precip_mm,has_precipitation,date,holiday_date,holiday_name,holiday_date_type,is_holiday,is_day_before_holiday,is_day_after_holiday,hour,dayofweek,dayofyear,weekofyear,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,trip_count_lag_1,trip_count_lag_2,trip_count_lag_3,trip_count_lag_24,trip_count_lag_48,trip_count_lag_72,trip_count_lag_168,trip_count_rolling_24_mean,trip_count_rolling_168_mean
0,2023-01-01 00:00:00,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,11.700000,10.60,1007.2,3.10,4828.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,0,6,1,52,1,1,0.000000,1.000000e+00,-0.781831,0.62349,0.5,0.866025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-01 01:00:00,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,11.133333,10.00,1007.2,3.10,6964.666667,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,1,6,1,52,1,1,0.258819,9.659258e-01,-0.781831,0.62349,0.5,0.866025,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-01 02:00:00,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,9.400000,8.30,1008.1,2.60,6437.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,2,6,1,52,1,1,0.500000,8.660254e-01,-0.781831,0.62349,0.5,0.866025,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-01 03:00:00,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,8.900000,7.80,1008.4,2.10,8047.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,3,6,1,52,1,1,0.707107,7.071068e-01,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-01 04:00:00,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,9.400000,8.30,1008.9,3.60,8047.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,4,6,1,52,1,1,0.866025,5.000000e-01,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-01-01 05:00:00,1,1.0,118.11,0.0,116.0,0.066667,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,8.900000,7.80,1009.8,4.60,11265.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,5,6,1,52,1,1,0.965926,2.588190e-01,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
6,2023-01-01 06:00:00,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,8.900000,7.80,1010.6,4.10,12875.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,6,6,1,52,1,1,1.000000,6.123234e-17,-0.781831,0.62349,0.5,0.866025,1.0,0.0,0.0,NaN,NaN,NaN,NaN,0.166667,NaN
7,2023-01-01 07:00:00,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,8.600000,7.25,1011.1,4.35,11632.500000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,7,6,1,52,1,1,0.965926,-2.588190e-01,-0.781831,0.62349,0.5,0.866025,0.0,1.0,0.0,NaN,NaN,NaN,NaN,0.142857,NaN
8,2023-01-01 08:00:00,1,1.0,100.00,0.0,89.0,0.116667,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,10.600000,7.80,1012.1,2.60,16093.000000,0.0,0,2023-01-01,2023-01-01,New Year's Day,actual,1,1,0,8,6,1,52,1,1,0.866025,-5.000000e-01,-0.781831,0.62349,0.5,0.866025,0.0,0.0,1.0,NaN,NaN,NaN,NaN,0.125000,NaN
9,2023-01-01 09:00:00,1,0.0,0.00,NaN,NaN,NaN,0.0,EWR,Newark Airport,EWR,newark,"Newark Liberty International Airport, NJ US",72502014734,725020,14734,12.200000,6.10,

## 3. Rebuild modeling features

I use the same safe feature set from model selection. Same-hour post-trip columns such as revenue, average fare, and average trip duration are excluded because they would not be known before making a future demand prediction.

In [3]:
target = "trip_count"

numeric_features = [
    "PULocationID",
    "temp_c",
    "dew_c",
    "pressure_hpa",
    "wind_speed_mps",
    "visibility_m",
    "precip_mm",
    "has_precipitation",
    "is_holiday",
    "is_day_before_holiday",
    "is_day_after_holiday",
    "hour",
    "dayofweek",
    "dayofyear",
    "weekofyear",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "trip_count_lag_1",
    "trip_count_lag_2",
    "trip_count_lag_3",
    "trip_count_lag_24",
    "trip_count_lag_48",
    "trip_count_lag_72",
    "trip_count_lag_168",
    "trip_count_rolling_24_mean",
    "trip_count_rolling_168_mean",
]

categorical_features = [
    "Borough",
    "service_zone",
    "weather_station_key",
    "holiday_name",
]

feature_cols = numeric_features + categorical_features
required_lag_cols = ["trip_count_lag_168", "trip_count_rolling_24_mean", "trip_count_rolling_168_mean"]

model_data = df[["pickup_hour", target] + feature_cols].dropna(subset=required_lag_cols).reset_index(drop=True)

print(model_data.shape)
model_data.head()

(2259696, 38)


,pickup_hour,trip_count,PULocationID,temp_c,dew_c,pressure_hpa,wind_speed_mps,visibility_m,precip_mm,has_precipitation,is_holiday,is_day_before_holiday,is_day_after_holiday,hour,dayofweek,dayofyear,weekofyear,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,trip_count_lag_1,trip_count_lag_2,trip_count_lag_3,trip_count_lag_24,trip_count_lag_48,trip_count_lag_72,trip_count_lag_168,trip_count_rolling_24_mean,trip_count_rolling_168_mean,Borough,service_zone,weather_station_key,holiday_name
0,2023-01-08 00:00:00,0.0,1,3.30,-3.9,1024.5,2.60,16093.0,0.0,0,0,0,0,0,6,8,1,1,1,0.000000,1.000000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,EWR,EWR,newark,None
1,2023-01-08 01:00:00,0.0,1,3.05,-3.9,1024.6,2.35,16046.5,0.0,0,0,0,0,1,6,8,1,1,1,0.258819,0.965926,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,EWR,EWR,newark,None
2,2023-01-08 02:00:00,0.0,1,2.20,-3.9,1025.4,3.10,16093.0,0.0,0,0,0,0,2,6,8,1,1,1,0.500000,0.866025,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,EWR,EWR,newark,None
3,2023-01-08 03:00:00,0.0,1,2.20,-3.9,1025.6,3.60,16093.0,0.0,0,0,0,0,3,6,8,1,1,1,0.707107,0.707107,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,EWR,EWR,newark,None
4,2023-01-08 04:00:00,0.0,1,2.20,-5.6,1026.2,3.60,16093.0,0.0,0,0,0,0,4,6,8,1,1,1,0.866025,0.500000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.25,0.738095,EWR,EWR,newark,None


## 4. Encode categorical columns and clean feature names

LightGBM requires numeric feature matrices. One-hot encoding converts small categorical columns into numeric indicator columns. Feature names are sanitized because LightGBM rejects special JSON characters in column names.

In [ ]:
def clean_feature_names(columns):
    """Sanitize feature names so LightGBM can use the encoded feature matrix"""
    cleaned = (
        pd.Series(columns)
        .str.replace(r"[^A-Za-z0-9_]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    counts = {}
    unique_names = []
    for name in cleaned:
        if name not in counts:
            counts[name] = 0
            unique_names.append(name)
        else:
            counts[name] += 1
            unique_names.append(f"{name}_{counts[name]}")
    return unique_names


def build_feature_matrix(input_df, feature_columns=None):
    """Create the numeric model matrix and optionally align it to a saved feature schema."""
    X = pd.get_dummies(
        input_df[feature_cols],
        columns=categorical_features,
        dummy_na=False,
        dtype=np.int8,
    )
    X.columns = clean_feature_names(X.columns)

    # during inference, new data must be aligned to the exact training feature schema
    if feature_columns is not None:
        X = X.reindex(columns=feature_columns, fill_value=0)

    return X


X_all = build_feature_matrix(model_data)
y_all = model_data[target].astype(float)

print(X_all.shape)
X_all.head()

(2259696, 57)


,PULocationID,temp_c,dew_c,pressure_hpa,wind_speed_mps,visibility_m,precip_mm,has_precipitation,is_holiday,is_day_before_holiday,is_day_after_holiday,hour,dayofweek,dayofyear,weekofyear,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,trip_count_lag_1,trip_count_lag_2,trip_count_lag_3,trip_count_lag_24,trip_count_lag_48,trip_count_lag_72,trip_count_lag_168,trip_count_rolling_24_mean,trip_count_rolling_168_mean,Borough_Bronx,Borough_Brooklyn,Borough_EWR,Borough_Manhattan,Borough_Queens,Borough_Staten_Island,service_zone_Airports,service_zone_Boro_Zone,service_zone_EWR,service_zone_Yellow_Zone,weather_station_key_central_park,weather_station_key_jfk,weather_station_key_laguardia,weather_station_key_newark,holiday_name_Birthday_of_Martin_Luther_King_Jr,holiday_name_Christmas_Day,holiday_name_Columbus_Day,holiday_name_Independence_Day,holiday_name_Juneteenth_National_Independence_Day,holiday_name_Labor_Day,holiday_name_Memorial_Day,holiday_name_None,holiday_name_Thanksgiving_Day,holiday_name_Veterans_Day,holiday_name_Washington_s_Birthday
0,1,3.30,-3.9,1024.5,2.60,16093.0,0.0,0,0,0,0,0,6,8,1,1,1,0.000000,1.000000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
1,1,3.05,-3.9,1024.6,2.35,16046.5,0.0,0,0,0,0,1,6,8,1,1,1,0.258819,0.965926,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
2,1,2.20,-3.9,1025.4,3.10,16093.0,0.0,0,0,0,0,2,6,8,1,1,1,0.500000,0.866025,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
3,1,2.20,-3.9,1025.6,3.60,16093.0,0.0,0,0,0,0,3,6,8,1,1,1,0.707107,0.707107,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
4,1,2.20,-5.6,1026.2,3.60,16093.0,0.0,0,0,0,0,4,6,8,1,1,1,0.866025,0.500000,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.25,0.738095,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0


## 5. Create chronological splits

I keep the same time-based structure:

- Jan-Oct: training for tuning
- November: validation for tuning/model improvement
- December: final held-out test and prediction example

In [5]:
train_mask = model_data["pickup_hour"] < "2023-11-01"
valid_mask = (model_data["pickup_hour"] >= "2023-11-01") & (model_data["pickup_hour"] < "2023-12-01")
test_mask = model_data["pickup_hour"] >= "2023-12-01"

X_train_full = X_all.loc[train_mask].copy()
X_valid = X_all.loc[valid_mask].copy()
X_test = X_all.loc[test_mask].copy()

y_train_full = y_all.loc[train_mask].copy()
y_valid = y_all.loc[valid_mask].copy()
y_test = y_all.loc[test_mask].copy()

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train_full), len(X_valid), len(X_test)],
        "start": [model_data.loc[train_mask, "pickup_hour"].min(), model_data.loc[valid_mask, "pickup_hour"].min(), model_data.loc[test_mask, "pickup_hour"].min()],
        "end": [model_data.loc[train_mask, "pickup_hour"].max(), model_data.loc[valid_mask, "pickup_hour"].max(), model_data.loc[test_mask, "pickup_hour"].max()],
        "total_trips": [y_train_full.sum(), y_valid.sum(), y_test.sum()],
    }
)

split_summary

,split,rows,start,end,total_trips
0,train,1874664,2023-01-08,2023-10-31 23:00:00,30112143.0
1,validation,189360,2023-11-01,2023-11-30 23:00:00,3249632.0
2,test,195672,2023-12-01,2023-12-31 23:00:00,3292743.0


## 6. Define metrics

I optimize using WAPE because it is easy to interpret for demand forecasting: total absolute error divided by total actual demand.

In [6]:
def rmse(y_true, y_pred):
    """Calculate root mean squared error for regression predictions."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def wape(y_true, y_pred):
    """Calculate weighted absolute percentage error for total demand forecasting."""
    denominator = np.sum(np.abs(y_true))
    if denominator == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denominator


def evaluate_regression(y_true, y_pred):
    """Return MAE, RMSE, WAPE, and R2 for a set of predictions."""
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "WAPE": wape(np.asarray(y_true), y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def predict_from_log_model(model, X):
    """Predict with a log-trained model and convert results back to trip counts."""
    return np.clip(np.expm1(model.predict(X)), 0, None)

## 7. Fine-tune LightGBM

This is a small manual hyperparameter search. Each candidate is trained on a reproducible sample from Jan-Oct and evaluated on the full November validation set.

The target is trained on `log1p(trip_count)` to reduce the influence of extreme high-demand zone-hours. Predictions are converted back with `expm1` before evaluation.

In [7]:
RANDOM_STATE = 42
TUNING_TRAIN_SAMPLE_ROWS = 400_000

# Sampling keeps tuning fast while the full November validation set remains unchanged
if len(X_train_full) > TUNING_TRAIN_SAMPLE_ROWS:
    tuning_index = y_train_full.sample(n=TUNING_TRAIN_SAMPLE_ROWS, random_state=RANDOM_STATE).index
    X_tune_train = X_train_full.loc[tuning_index]
    y_tune_train = y_train_full.loc[tuning_index]
else:
    X_tune_train = X_train_full
    y_tune_train = y_train_full

param_candidates = [
    {"n_estimators": 700, "learning_rate": 0.05, "num_leaves": 64, "min_child_samples": 40, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.0, "reg_lambda": 0.2},
    {"n_estimators": 900, "learning_rate": 0.04, "num_leaves": 96, "min_child_samples": 50, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.0, "reg_lambda": 0.5},
    {"n_estimators": 1000, "learning_rate": 0.03, "num_leaves": 128, "min_child_samples": 60, "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 0.1, "reg_lambda": 0.5},
    {"n_estimators": 800, "learning_rate": 0.06, "num_leaves": 48, "min_child_samples": 30, "subsample": 0.80, "colsample_bytree": 0.90, "reg_alpha": 0.0, "reg_lambda": 0.1},
    {"n_estimators": 1000, "learning_rate": 0.035, "num_leaves": 160, "min_child_samples": 80, "subsample": 0.90, "colsample_bytree": 0.80, "reg_alpha": 0.2, "reg_lambda": 1.0},
    {"n_estimators": 750, "learning_rate": 0.045, "num_leaves": 80, "min_child_samples": 35, "subsample": 0.90, "colsample_bytree": 0.85, "reg_alpha": 0.05, "reg_lambda": 0.3},
]

tuning_results = []
best_model = None
best_wape = np.inf
best_params = None

for i, params in enumerate(param_candidates, start=1):
    print(f"Tuning candidate {i}/{len(param_candidates)}")
    start = time.time()

    model = LGBMRegressor(
        objective="regression",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        **params,
    )

    model.fit(
        X_tune_train,
        np.log1p(y_tune_train),
        eval_set=[(X_valid, np.log1p(y_valid))],
        eval_metric="rmse",
        callbacks=[early_stopping(50, verbose=False), log_evaluation(period=0)],
    )

    pred_valid = predict_from_log_model(model, X_valid)
    metrics = evaluate_regression(y_valid, pred_valid)
    metrics.update(params)
    metrics["best_iteration"] = getattr(model, "best_iteration_", params["n_estimators"])
    metrics["TrainSeconds"] = time.time() - start
    tuning_results.append(metrics)

    if metrics["WAPE"] < best_wape:
        best_wape = metrics["WAPE"]
        best_model = model
        best_params = params.copy()

tuning_results_df = pd.DataFrame(tuning_results).sort_values("WAPE").reset_index(drop=True)
tuning_results_df.to_csv(TUNING_RESULTS_PATH, index=False)
tuning_results_df

Tuning candidate 1/6
Tuning candidate 2/6
Tuning candidate 3/6
Tuning candidate 4/6
Tuning candidate 5/6
Tuning candidate 6/6


,MAE,RMSE,WAPE,R2,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda,best_iteration,TrainSeconds
0,2.890458,10.293901,0.168430,0.963799,1000,0.035,160,80,0.90,0.80,0.20,1.0,630,57.939957
1,2.925858,10.502733,0.170493,0.962315,900,0.040,96,50,0.85,0.85,0.00,0.5,399,28.258789
2,2.934173,10.517265,0.170978,0.962211,1000,0.030,128,60,0.90,0.90,0.10,0.5,533,40.364044
3,2.951561,10.535575,0.171991,0.962079,750,0.045,80,35,0.90,0.85,0.05,0.3,449,36.482332
4,2.954114,10.540353,0.172140,0.962045,700,0.050,64,40,0.85,0.85,0.00,0.2,578,40.154561
5,2.980412,10.660122,0.173672,0.961177,800,0.060,48,30,0.80,0.90,0.00,0.1,317,17.416562


## 8. Train final model on Jan-Nov

After choosing the best parameters using November validation data, I train the final model on January-November and evaluate on December.

In [8]:
best_iteration = int(tuning_results_df.loc[0, "best_iteration"])
best_params = {
    key: tuning_results_df.loc[0, key]
    for key in ["learning_rate", "num_leaves", "min_child_samples", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda"]
}
best_params["n_estimators"] = max(best_iteration, 100)
best_params["num_leaves"] = int(best_params["num_leaves"])
best_params["min_child_samples"] = int(best_params["min_child_samples"])

X_train_valid = pd.concat([X_train_full, X_valid], axis=0)
y_train_valid = pd.concat([y_train_full, y_valid], axis=0)

final_model = LGBMRegressor(
    objective="regression",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
    **best_params,
)

start = time.time()
final_model.fit(X_train_valid, np.log1p(y_train_valid))
final_train_seconds = time.time() - start

test_pred = predict_from_log_model(final_model, X_test)
test_metrics = evaluate_regression(y_test, test_pred)
test_metrics["TrainSeconds"] = final_train_seconds

pd.DataFrame([test_metrics])

,MAE,RMSE,WAPE,R2,TrainSeconds
0,3.770455,13.35341,0.22406,0.934903,131.046047


## 9. Save the final model artifact

The saved artifact includes the model and the exact feature schema. The feature schema is important because inference data must have the same columns in the same order as training data.

In [9]:
model_artifact = {
    "model": final_model,
    "target": target,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "feature_cols": feature_cols,
    "feature_columns": X_all.columns.tolist(),
    "best_params": best_params,
    "validation_metrics": tuning_results_df.iloc[0].to_dict(),
    "test_metrics": test_metrics,
    "training_note": "Model trained on Jan-Nov 2023 and evaluated on Dec 2023.",
}

joblib.dump(model_artifact, MODEL_ARTIFACT_PATH)

MODEL_ARTIFACT_PATH

WindowsPath('c:/Users/USER/Downloads/calibrax/forecasting-taxi-demand-across-nyc-zones/data/processed/final_lgbm_trip_demand_model.joblib')

## 10. Reusable inference function

This function expects rows that have already gone through the preparation process. In deployment, future rows must include future calendar/holiday/weather features and lag features built from recent historical demand.

In [ ]:
def load_model_artifact(path=MODEL_ARTIFACT_PATH):
    """Load the saved model artifact from disk."""
    return joblib.load(path)


def predict_taxi_demand(input_rows, artifact_path=MODEL_ARTIFACT_PATH):
    """predict hourly taxi demand for prepared rows using the saved model artifact"""
    artifact = load_model_artifact(artifact_path)

    X = build_feature_matrix(
        input_rows,
        feature_columns=artifact["feature_columns"],
    )

    predictions = np.clip(np.expm1(artifact["model"].predict(X)), 0, None)

    output = input_rows[["pickup_hour", "PULocationID"]].copy()
    output["predicted_trip_count"] = predictions

    if target in input_rows.columns:
        output["actual_trip_count"] = input_rows[target].values
        output["absolute_error"] = (output["actual_trip_count"] - output["predicted_trip_count"]).abs()

    return output


sample_predictions = predict_taxi_demand(model_data.loc[test_mask].head(10))
sample_predictions

,pickup_hour,PULocationID,predicted_trip_count,actual_trip_count,absolute_error
7848,2023-12-01 00:00:00,1,0.063969,0.0,0.063969
7849,2023-12-01 01:00:00,1,0.063433,0.0,0.063433
7850,2023-12-01 02:00:00,1,0.062217,0.0,0.062217
7851,2023-12-01 03:00:00,1,0.064455,1.0,0.935545
7852,2023-12-01 04:00:00,1,0.370274,0.0,0.370274
7853,2023-12-01 05:00:00,1,0.212704,0.0,0.212704
7854,2023-12-01 06:00:00,1,0.301057,0.0,0.301057
7855,2023-12-01 07:00:00,1,0.384254,0.0,0.384254
7856,2023-12-01 08:00:00,1,0.270072,0.0,0.270072
7857,2023-12-01 09:00:00,1,0.305988,1.0,0.694012


## 11. Save December predictions

This creates a prediction file for the held-out December period. 

In [11]:
december_predictions = predict_taxi_demand(model_data.loc[test_mask])
december_predictions.to_parquet(TEST_PREDICTIONS_PATH, index=False)

print(TEST_PREDICTIONS_PATH)
december_predictions.head()

c:\Users\USER\Downloads\calibrax\forecasting-taxi-demand-across-nyc-zones\data\processed\december_lgbm_predictions.parquet


,pickup_hour,PULocationID,predicted_trip_count,actual_trip_count,absolute_error
7848,2023-12-01 00:00:00,1,0.063969,0.0,0.063969
7849,2023-12-01 01:00:00,1,0.063433,0.0,0.063433
7850,2023-12-01 02:00:00,1,0.062217,0.0,0.062217
7851,2023-12-01 03:00:00,1,0.064455,1.0,0.935545
7852,2023-12-01 04:00:00,1,0.370274,0.0,0.370274


## 12. Summarize predictions by taxi zone

The prediction file is hourly, so I aggregate it here to show total predicted December demand for each pickup zone.


In [12]:
ZONE_PREDICTION_SUMMARY_PATH = PROCESSED_DIR / "december_lgbm_predictions_by_zone.csv"

zone_lookup = pd.read_csv(PROJECT_DIR / "data" / "taxi_zone_lookup.csv")

zone_prediction_summary = (
    december_predictions.groupby("PULocationID", as_index=False)
    .agg(
        total_predicted_trips=("predicted_trip_count", "sum"),
        total_actual_trips=("actual_trip_count", "sum"),
        total_absolute_error=("absolute_error", "sum"),
    )
    .merge(zone_lookup, left_on="PULocationID", right_on="LocationID", how="left")
    .drop(columns=["LocationID"])
)

zone_prediction_summary["zone_wape"] = (
    zone_prediction_summary["total_absolute_error"]
    / zone_prediction_summary["total_actual_trips"].replace(0, np.nan)
)

zone_prediction_summary = zone_prediction_summary.sort_values(
    "total_predicted_trips",
    ascending=False,
).reset_index(drop=True)

zone_prediction_summary.to_csv(ZONE_PREDICTION_SUMMARY_PATH, index=False)

zone_prediction_summary.head(20)

,PULocationID,total_predicted_trips,total_actual_trips,total_absolute_error,Borough,Zone,service_zone,zone_wape
0,237,190415.040176,162336.0,32624.876730,Manhattan,Upper East Side South,Yellow Zone,0.200971
1,236,173635.554413,147654.0,30976.023187,Manhattan,Upper East Side North,Yellow Zone,0.209788
2,161,170212.401080,143545.0,31191.202395,Manhattan,Midtown Center,Yellow Zone,0.217292
3,132,164385.881122,141476.0,31025.656557,Queens,JFK Airport,Airports,0.219300
4,162,141125.471445,118175.0,26049.259924,Manhattan,Midtown East,Yellow Zone,0.220430
5,142,136145.230134,117822.0,27494.566586,Manhattan,Lincoln Square East,Yellow Zone,0.233357
6,186,127636.798187,110183.0,25874.821168,Manhattan,Penn Station/Madison Sq West,Yellow Zone,0.234835
7,230,127162.380949,109480.0,23087.804020,Manhattan,Times Sq/Theatre District,Yellow Zone,0.210886
8,138,121024.597941,101634.0,27805.054253,Queens,LaGuardia Airport,Airports,0.273580
9,163,117567.338105,100308.0,20584.395564,Manhattan,Midtown North,Yellow Zone,0.205212


## 13. Summary for report

Latest run results:

- Best validation WAPE after tuning: `0.1684`
- Best validation MAE after tuning: `2.8905`
- Best validation RMSE after tuning: `10.2939`
- Final December test WAPE: `0.2241`
- Final December test MAE: `3.7705`
- Final December test RMSE: `13.3534`
- Final December test R2: `0.9349`

The validation result improved slightly compared with the earlier untuned LightGBM validation WAPE of about `0.1696`. December performance is weaker than November validation performance, which suggests the final month has harder holiday/year-end demand patterns.

